In [ ]:
!pip install diffusers transformers accelerate torch safetensors bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


# Clean the GPU in Google Colab

In [ ]:
import torch
import gc

# 1. Clear the Python variables
if 'model' in globals():
    del model
if 'pipe' in globals():
    del pipe

# 2. Force Garbage Collection
gc.collect()

# 3. Empty the CUDA (GPU) Cache
torch.cuda.empty_cache()

print("GPU Memory Cleared!")

GPU Memory Cleared!


# Sample transformer

In [ ]:
from transformers import pipeline		#dùng thư viện transformer của HuggingFace
import pandas as pd #thư viện xử lý sheet

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") #thư viện phân loại câu của Facebook
reviews = [ #các câu cần phân loại
   "I bought the Galaxy S24 yesterday. The screen is beautiful but the battery dies in 2 hours. Very disappointed.",
   "The new MacBook is so fast! I love the keyboard, though it was quite expensive.",
   "Avoid this blender. It started smoking the first time I made a smoothie. Dangerous!"
]
candidate_labels = ["electronics", "kitchen", "positive", "negative", "safety-hazard"]  #các loại tiêu biểu
results = []
for text in reviews:  #lặp từng câu, tìm từ phù hợp nhất của câu đó
   prediction = classifier(text, candidate_labels)
   top_label = prediction['labels'][0]
   score = prediction['scores'][0]  #điểm liên quan giữa câu và loại câu
   results.append({
       "Review": text[:30] + "...",
       "Category/Sentiment": top_label,
       "Confidence": f"{score:.2%}"
   })
df = pd.DataFrame(results)  #~30 secs
print(df)     #in kết quả


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

                              Review Category/Sentiment Confidence
0  I bought the Galaxy S24 yester...           negative     77.51%
1  The new MacBook is so fast! I ...        electronics     51.61%
2  Avoid this blender. It started...      safety-hazard     65.77%


# Generate text

In [ ]:
# 1. Load a light 'Text-to-Text' generation model (Flan-T5)
# This model is very efficient and fits on a standard laptop.
generator = pipeline("text-generation", model="google/flan-t5-small")

# 2. Define a function to handle the "Ask & Answer" logic
def ask_ai(question, context=""):
    # We combine the question and context into a 'Prompt'
    if context:
        input_text = f"Context: {context} Question: {question}"
    else:
        input_text = f"Question: {question}"

    # The AI generates the answer based on the input
    output = generator(input_text, max_length=50, num_beams=5)
    return output[0]['generated_text']

# 3. Try it out!
# Example A: General Knowledge
question = 'Who is the president of USA?'
answer = ask_ai(question)
print(f"Q: {question}\nA: {answer}")

# Example B: Reading from Context (Like a mini-RAG)
# paragraph = "The company 'TechNova' was founded in 2024 in Ho Chi Minh City by a group of IT instructors."
# query = "Where and when was TechNova founded?"

# print(f"\nContext: {paragraph}")
# print(f"Q: {query}\nA: {ask_ai(query, paragraph)}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

Q: Who is the president of USA?
A: Question: Who is the president of USA?din.ca


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

model_id = "microsoft/Phi-3.5-mini-instruct"

# 2. Define the compression (Quantization) configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# 3. LOAD the model and tokenizer FIRST (This is where the config goes)
# Change trust_remote_code to False
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=False  # <--- Change this to False
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 4. Create the pipeline using the ALREADY LOADED model
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# 5. Use the pipeline (No config needed here)
messages = [{"role": "user", "content": "Who is the president of USA?"}]
output = pipe(messages, max_new_tokens=200)
print(output[0]['generated_text'][-1]['content'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 As of my knowledge cutoff in April 2023, the President of the United States is Joe Biden. He assumed office on January 20, 2021, after winning the presidential election in November 2020. Please note that political positions can change, and it is always a good idea to verify current officeholders from reliable sources.


In [ ]:
#try LLM with different persons
def ask_as_person(question, persona):
    messages = [
        {"role": "system", "content": f"You are {persona}. Answer the user accurately in your specific style."},
        {"role": "user", "content": question},
    ]
    output = pipe(messages, max_new_tokens=150, temperature=0.8)
    return output[0]['generated_text'][-1]['content']

# Classroom Demo:
print("Scientist:", ask_as_person("What is fire?", persona="a professional scientist"))
print("Pirate:", ask_as_person("What is fire?", persona="a pirate captain"))

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Scientist:  Fire, from a scientific perspective, is a rapid oxidation chemical reaction known as combustion. This process involves the exothermic reaction between a fuel (typically a combustible material) and an oxidizing agent, usually atmospheral oxygen for most occurrences, resulting in the release of energy in the form of heat and light, often visible as flames.

Fires manifest in various forms but commonly require three elements to sustain the process: heat (ignition source), fuel, and oxygen. This triad is often referred to as the "fire triangle."

1. Heat (Ignition Source): The heat provides the necessary activation energy to initiate the combustion process
Pirate:  Ahoy there, me hearty! I be the captain o' this fine ship, but I can surely tell ye 'bout fire, a wonder of nature that be both a blessing and a curse aboard a pirate vessel.

Fire, me matey, be a fierce and relentless reaction 'twixt two things that be oxygen, a gas we pirates call "air", and fuel, like wood or oil.

# Generate image

In [ ]:

from diffusers import DiffusionPipeline
import torch

#image
pipe = DiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", #~2.4GB approx. 7mins to load
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe = pipe.to("cuda")


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def generate_img(prompt, output_filename):
  image = pipe(prompt).images[0]
  image.save(output_filename)
  image #show the image

In [ ]:
prompt='a cat is chasing a mouse on the wood floor'
generate_img(prompt, 'output2.png')

  0%|          | 0/50 [00:00<?, ?it/s]

# Store image in Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Generate video

In [ ]:
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
import torch

# Load ONLY the Zeroscope model
pipe = DiffusionPipeline.from_pretrained(
    "cerspense/zeroscope_v2_576w",  #3.7GB
    torch_dtype=torch.float16
).to("cuda")

# Optimization for Colab Free (Saves VRAM)
pipe.enable_model_cpu_offload()

# Setting the scheduler for smoother motion
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Generate
prompt = "a robot teaching students in a futuristic classroom, cinematic"
video_frames = pipe(prompt, num_inference_steps=25, num_frames=24).frames

model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/s

  0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
import imageio
import numpy as np

# 1. Convert to a single numpy array if it isn't one
video_np = np.array(video_frames)

# 2. Fix the Shape: We need (Frames, Height, Width, Channels)
# If the AI gave us (Frames, Channels, Height, Width), we swap them
if video_np.shape[1] == 3: # Check if '3' is in the second slot
    video_np = video_np.transpose(0, 2, 3, 1)

# 3. Remove "Empty" dimensions (like a 1 at the start)
video_np = np.squeeze(video_np)

# 4. Normalize to 0-255 (uint8)
if video_np.max() <= 1.0:
    video_np = (video_np * 255).astype(np.uint8)
else:
    video_np = video_np.astype(np.uint8)

# 5. Debug Print for the class
print(f"Final Video Shape for saving: {video_np.shape}")

# 6. Save
imageio.mimsave("video_2.mp4", video_np, fps=10)

Final Video Shape for saving: (24, 256, 256, 3)


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')